# LLM Zoomcamp 2026 - dlt Workshop Homework

Notebook para ejecutar la tarea por partes y conservar las salidas visibles.

Antes de correrlo, configura `.env` con:

```bash
OPENAI_API_KEY=...
LOGFIRE_TOKEN=...
LOGFIRE_READ_TOKEN=...
```


## 0. Dependencies

Ejecuta esta celda solo si te faltan paquetes en el entorno del notebook.


In [ ]:
# %pip install openai minsearch requests python-dotenv pydantic-ai logfire duckdb 'dlt[duckdb]' 'dlt[rest_api]'


## 1. Imports and Configuration


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from typing import Any

import duckdb
import requests
from dotenv import load_dotenv
from minsearch import Index
from pydantic_ai import Agent, RunContext

load_dotenv()

HOMEWORK_QUESTION = 'How do I run Ollama locally?'
DUCKDB_PATH = r'C:\Users\Valentina Cruz DP\Documents\Camila_bootcamp\llm-zoomcamp-2026-code\zoompcamp-h5\logfire.duckdb'
DATASET_NAME = 'agent_traces'

def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'Missing required environment variable: {name}')
    return value

{name: bool(os.getenv(name)) for name in ['OPENAI_API_KEY', 'LOGFIRE_TOKEN', 'LOGFIRE_READ_TOKEN']}


## 2. Agent Setup


In [ ]:
INSTRUCTIONS = '''
You're a course teaching assistant. You're given a question from a course student
and your task is to answer it. If you want to look up information, use the search
function. Use as many keywords from the user question as possible when making
first requests. Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic
question. If you can't answer the question using FAQ, don't do it yourself. Only
use the facts from the FAQ database. At the end, ask if there are other areas
that the user wants to explore.
'''.strip()

@dataclass
class SearchDeps:
    index: Index

faq_agent = Agent(
    'openai:gpt-5.4-mini',
    deps_type=SearchDeps,
    instructions=INSTRUCTIONS,
)

@faq_agent.tool
def search(ctx: RunContext[SearchDeps], query: str) -> str:
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}
    return ctx.deps.index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
    )


## 3. Load FAQ Data


In [ ]:
def load_faq_data() -> list[dict[str, Any]]:
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url, timeout=30)
    response.raise_for_status()

    documents: list[dict[str, Any]] = []
    url_prefix = 'https://datatalks.club/faq'
    for course in response.json():
        course_response = requests.get(f'{url_prefix}{course["path"]}', timeout=30)
        course_response.raise_for_status()
        documents.extend(course_response.json())

    return documents

def build_index(documents: list[dict[str, Any]]) -> Index:
    index = Index(
        text_fields=['question', 'section', 'answer'],
        keyword_fields=['course'],
    )
    index.fit(documents)
    return index

documents = load_faq_data()
deps = SearchDeps(index=build_index(documents))
len(documents)


## 4. Run Agent and Create Logfire Trace


In [ ]:
require_env('OPENAI_API_KEY')
require_env('LOGFIRE_TOKEN')

import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

result = await faq_agent.run(HOMEWORK_QUESTION, deps=deps)
print(result.output)

print('Question 1: open this run in Logfire and count spans.')


## 5. Load Logfire Traces into DuckDB with dlt


In [ ]:
import requests
import dlt
from dotenv import load_dotenv
load_dotenv()

LOGFIRE_BASE = 'https://logfire-us.pydantic.dev'
LOGFIRE_READ_TOKEN = os.getenv('LOGFIRE_READ_TOKEN')

def logfire_query_source(sql: str):
    @dlt.resource(name='query', write_disposition='replace')
    def query_resource():
        response = requests.post(
            f'{LOGFIRE_BASE}/v2/query',
            json={
                'sql': sql,
                'min_timestamp': '2020-01-01T00:00:00Z',
            },
            headers={
                'Authorization': f'Bearer {LOGFIRE_READ_TOKEN}',
                'Content-Type': 'application/json',
                'Accept': 'application/json',
            },
        )
        response.raise_for_status()
        data = response.json()['data']
        print(f'DEBUG: API devolvió {len(data)} rows')
        if data:
            print(f'DEBUG: columns: {list(data[0].keys())}')
        yield from data
    return query_resource

sql = '''
SELECT * FROM records
ORDER BY start_timestamp DESC
LIMIT 1000
'''.strip()

pipeline = dlt.pipeline(
    pipeline_name='logfire_pipeline',
    destination=dlt.destinations.duckdb(DUCKDB_PATH),
    dataset_name=DATASET_NAME,
)
load_info = pipeline.run(logfire_query_source(sql)())
print(load_info)


## 6. Question 2 - Count Tables


In [ ]:
con = duckdb.connect(DUCKDB_PATH)

table_count = con.sql(
    f'''
    SELECT COUNT(*)
    FROM information_schema.tables
    WHERE table_schema = '{DATASET_NAME}'
    '''
).fetchone()[0]

tables = con.sql(
    f'''
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = '{DATASET_NAME}'
    ORDER BY table_name
    '''
).fetchall()

print(f'Question 2 table count: {table_count}')
for (table_name,) in tables:
    print(table_name)


## 7. Question 3 - Sum Input Tokens


In [ ]:
candidates = con.sql(
    f'''
    SELECT table_schema, table_name, column_name
    FROM information_schema.columns
    WHERE table_schema = '{DATASET_NAME}'
      AND lower(column_name) LIKE '%input%token%'
    ORDER BY table_name, column_name
    '''
).fetchall()

candidates


In [ ]:
selects = []
for schema, table, column in candidates:
    selects.append(
        f'SELECT SUM(TRY_CAST("{column}" AS BIGINT)) AS tokens '
        f'FROM "{schema}"."{table}"'
    )

if not selects:
    print('No input-token column found in normalized tables.')
else:
    token_sql = 'SELECT SUM(tokens) FROM (' + ' UNION ALL '.join(selects) + ')'
    token_total = con.sql(token_sql).fetchone()[0]
    print(f'Question 3 input tokens total: {token_total}')


## 8. Final Answers


In [ ]:
answers = {
    'question_1': 5,
    'question_2': 24,
    'question_3': '1500 - 5000',
}
answers
